In [0]:
#dbutils.fs.mkdirs("/Workspace/dados")
#dbutils.fs.ls("/Workspace")


In [0]:
# Criando catálogo e schemas

spark.sql("CREATE CATALOG IF NOT EXISTS inbound")

spark.sql("CREATE SCHEMA IF NOT EXISTS inbound.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS inbound.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS inbound.gold")

# DEPOIS QUE FIZ UPLOAD RODEI ISTO AQUI

In [0]:
df = spark.read.json(
    "/Volumes/inbound/bronze/raw/dados_brutos_imoveis.json"
)

display(df)

### Salvar o bronze


In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("inbound.bronze.imoveis_raw")

### Depois você criará Silver

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS inbound.silver;

### Criar inbound gold


In [0]:
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS inbound.gold"
)

### Depois disso Você poderá salvar tabelas assim: 
Bronze

In [0]:
df.write.saveAsTable(
    "inbound.bronze.imoveis_raw"
)

#### Já existia, overwrite

In [0]:
df.write \
  .mode("overwrite") \
  .saveAsTable("inbound.bronze.imoveis_raw")

### Visualize

In [0]:
display(
    spark.table("inbound.bronze.imoveis_raw")
)

### Silver

limpa

In [0]:
df_silver = df.dropna()

cria silveer

In [0]:
from pyspark.sql.functions import col, get

df_silver = df.select(
    col("anuncio.id").alias("id_imovel"),
    col("anuncio.tipo_anuncio").alias("tipo_anuncio"),
    col("anuncio.tipo_unidade").alias("tipo_unidade"),
    col("anuncio.tipo_uso").alias("tipo_uso"),

    col("anuncio.endereco.cidade").alias("cidade"),
    col("anuncio.endereco.estado").alias("estado"),
    col("anuncio.endereco.bairro").alias("bairro"),
    col("anuncio.endereco.cep").alias("cep"),

    col("usuario.nome").alias("usuario"),

    get(col("anuncio.quartos"), 0).alias("quartos"),
    get(col("anuncio.banheiros"), 0).alias("banheiros"),
    get(col("anuncio.vaga"), 0).alias("vagas"),

    get(col("anuncio.area_total"), 0).alias("area_total"),

    get(col("anuncio.valores"), 0)["valor"].alias("valor")
)

visualize

In [0]:
display(df_silver)

apaga caso exista

In [0]:
spark.sql("""
DROP TABLE IF EXISTS inbound.silver.imoveis_tratado
""")

salva

In [0]:
df_silver.write \
    .mode("overwrite") \
    .saveAsTable("inbound.silver.imoveis_tratado")

visualiza

In [0]:
display(
    spark.table("inbound.silver.imoveis_tratado")
)

### Gold

In [0]:
from pyspark.sql.functions import avg, regexp_replace

df_gold = (
    df_silver
    .withColumn(
        "valor_num",
        regexp_replace("valor", "[^0-9]", "").cast("double")
    )
    .groupBy("cidade")
    .agg(
        avg("valor_num").alias("preco_medio")
    )
)

salvar gold

In [0]:
df_gold.write \
    .mode("overwrite") \
    .saveAsTable("inbound.gold.preco_medio_cidade")

visualizar gold

In [0]:
display(df_gold)

visualizar schema (isto poderia ser feito lá no começo)

In [0]:
df.printSchema()

# Listar dbfs

In [0]:
display(
    dbutils.fs.ls("/Volumes/inbound/bronze/raw")
)

caminho completo

In [0]:
for f in dbutils.fs.ls("/Volumes/inbound/bronze/raw"):
    print(f.path)

ler json

In [0]:
df = spark.read.json(
    "/Volumes/inbound/bronze/raw/dados_brutos_imoveis.json"
)

ler csv

In [0]:
df = spark.read.csv(
    "/Volumes/inbound/bronze/raw/arquivo.csv",
    header=True
)

Databricks moderno

/Volumes/

Unity Catalog

Managed Volumes